# Phase 1: Data Cleaning & Relational Database Setup
- **Objective:** Load the 6 relational CSV files, handle missing/zero values, remove duplicates and orphaned foreign keys, and load the cleaned tables into an SQLite database (`tmdb_movies.db`).

In [4]:
import pandas as pd
import os

# 1. Create local directories in Colab session
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/database', exist_ok=True)

# 2. Upload files prompt (Files-a upload panrathukana code)
from google.colab import files
uploaded = files.upload()

# Move uploaded files to /content/data/ directory
for filename in uploaded.keys():
    os.rename(filename, os.path.join('/content/data', filename))
    print(f"Moved {filename} to /content/data/")

# 3. Load all 6 CSV files into Pandas DataFrames
DATA_DIR = '/content/data/'
movies_df = pd.read_csv(DATA_DIR + 'movies.csv')
genres_df = pd.read_csv(DATA_DIR + 'genres.csv')
movie_genres_df = pd.read_csv(DATA_DIR + 'movie_genres.csv')
cast_df = pd.read_csv(DATA_DIR + 'cast.csv')
crew_df = pd.read_csv(DATA_DIR + 'crew.csv')
keywords_df = pd.read_csv(DATA_DIR + 'movie_keywords.csv')

# Print initial shapes to verify
print("\nInitial shapes loaded successfully:")
print(f"Movies: {movies_df.shape}, Genres: {genres_df.shape}")
print(f"Cast: {cast_df.shape}, Crew: {crew_df.shape}, Keywords: {keywords_df.shape}")

Saving cast.csv to cast.csv
Saving crew.csv to crew.csv
Saving genres.csv to genres.csv
Saving movie_genres.csv to movie_genres.csv
Saving movie_keywords.csv to movie_keywords.csv
Saving movies.csv to movies.csv
Moved cast.csv to /content/data/
Moved crew.csv to /content/data/
Moved genres.csv to /content/data/
Moved movie_genres.csv to /content/data/
Moved movie_keywords.csv to /content/data/
Moved movies.csv to /content/data/

Initial shapes loaded successfully:
Movies: (2509, 12), Genres: (19, 2)
Cast: (24917, 6), Crew: (7297, 6), Keywords: (41701, 3)


### Data Cleaning & Type Casting Logic:
- Parsing `release_date` to proper datetime format.
- Ensuring ID columns are integers.
- Dropping exact duplicate rows across all tables.
- Removing orphaned rows (e.g., cast or genre rows pointing to a `movie_id` that doesn't exist in `movies.csv`).

In [5]:
# 1. Type Casting & Date Parsing
movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')
movies_df['movie_id'] = movies_df['movie_id'].astype('Int64')
genres_df['genre_id'] = genres_df['genre_id'].astype('Int64')
movie_genres_df['movie_id'] = movie_genres_df['movie_id'].astype('Int64')
movie_genres_df['genre_id'] = movie_genres_df['genre_id'].astype('Int64')

# 2. Remove exact duplicate rows[cite: 1]
movies_df.drop_duplicates(inplace=True)
genres_df.drop_duplicates(inplace=True)
movie_genres_df.drop_duplicates(inplace=True)
cast_df.drop_duplicates(inplace=True)
crew_df.drop_duplicates(inplace=True)
keywords_df.drop_duplicates(inplace=True)

# 3. Remove Orphaned Rows (Foreign key validation)[cite: 1]
valid_movie_ids = set(movies_df['movie_id'])
valid_genre_ids = set(genres_df['genre_id'])

movie_genres_df = movie_genres_df[movie_genres_df['movie_id'].isin(valid_movie_ids)]
movie_genres_df = movie_genres_df[movie_genres_df['genre_id'].isin(valid_genre_ids)]
cast_df = cast_df[cast_df['movie_id'].isin(valid_movie_ids)]
crew_df = crew_df[crew_df['movie_id'].isin(valid_movie_ids)]
keywords_df = keywords_df[keywords_df['movie_id'].isin(valid_movie_ids)]

print("Shapes after cleaning and removing orphans:")
print(f"Movies: {movies_df.shape}, Movie-Genres: {movie_genres_df.shape}")

Shapes after cleaning and removing orphans:
Movies: (2503, 12), Movie-Genres: (6878, 2)


In [7]:
import sqlite3

# 1. Load all cleaned tables into SQLite Database[cite: 1]
db_path = '/content/data/tmdb_movies.db'
conn = sqlite3.connect(db_path)

movies_df.to_sql('movies', conn, if_exists='replace', index=False)
genres_df.to_sql('genres', conn, if_exists='replace', index=False)
movie_genres_df.to_sql('movie_genres', conn, if_exists='replace', index=False)
cast_df.to_sql('cast', conn, if_exists='replace', index=False)
crew_df.to_sql('crew', conn, if_exists='replace', index=False)
keywords_df.to_sql('movie_keywords', conn, if_exists='replace', index=False)

# 2. Test query to confirm database loaded correctly[cite: 1]
test_query = """
    SELECT m.title, g.genre_name, m.release_date
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    LIMIT 5
"""
test_df = pd.read_sql(test_query, conn)
conn.close()

print("\n--- Phase 1 Completed Successfully! ---")
print(test_df)


--- Phase 1 Completed Successfully! ---
          title       genre_name         release_date
0  Interstellar        Adventure  2014-11-05 00:00:00
1  Interstellar            Drama  2014-11-05 00:00:00
2  Interstellar  Science Fiction  2014-11-05 00:00:00
3     Inception        Adventure  2010-07-15 00:00:00
4     Inception           Action  2010-07-15 00:00:00


### Written Insight & Recommendation:
- **Data Integrity:** Removing duplicate records and orphaned foreign keys ensures that subsequent SQL joins and Pandas aggregations do not produce inflated or erroneous counts.
- **Handling Zero-Values:** Budget and revenue values equal to zero in TMDB indicate missing or undisclosed data rather than free films. Preserving these as zeros (rather than deleting rows) allows us to isolate valid box-office figures accurately during financial and ROI analysis.